In [2]:
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline



In [4]:
!kaggle datasets download -d abhi8923shriv/sentiment-analysis-dataset


Dataset URL: https://www.kaggle.com/datasets/abhi8923shriv/sentiment-analysis-dataset
License(s): CC0-1.0
sentiment-analysis-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [5]:
!unzip sentiment-analysis-dataset.zip

Archive:  sentiment-analysis-dataset.zip
replace test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: test.csv                
replace testdata.manual.2009.06.14.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: testdata.manual.2009.06.14.csv  
replace train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: train.csv               
replace training.1600000.processed.noemoticon.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: training.1600000.processed.noemoticon.csv  


In [16]:
df['sentiment'].unique()

array(['neutral', 'negative', 'positive'], dtype=object)

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from collections import Counter
import numpy as np

In [18]:
data = [(df['selected_text'][i], df['sentiment'][i]) for i in range(len(df))]

In [19]:
sentences, labels = zip(*data)

In [20]:
len(sentences), len(labels)

(27480, 27480)

In [21]:
def tokenize(text):
  if isinstance(text, str):
    return text.lower().split()
  return [] # Handle non-string values (e.g., NaN) by returning an empty list


vocab = Counter()
for sentence in sentences:
  vocab.update(tokenize(sentence))
vocab = {word: idx+1 for idx, (word, _) in enumerate(vocab.most_common())}



def encode_sentence(sentence, vocab):
  # Ensure sentence is a string before tokenizing
  if not isinstance(sentence, str):
    return [] # Return empty for non-string inputs
  return [vocab.get(word, 0) for word in tokenize(sentence)]

encoded_sentences = [encode_sentence(sentence, vocab) for sentence in sentences]

In [22]:
sentences, labels = zip(*data)
# Pad sequences to the same length
max_len = max(len(seq) for seq in encoded_sentences)
encoded_sentences = [seq + [0] * (max_len - len(seq)) for seq in encoded_sentences]

# Encoded labels for multi-class classification:
# Map 'negative' to 0, 'neutral' to 1, 'positive' to 2
label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
encoded_labels = np.array([label_mapping.get(label, -1) for label in labels ]) # Use .get with a default to handle potential missing labels gracefully

# Convert labels to long type for CrossEntropyLoss
encoded_labels = encoded_labels.astype(np.int64)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(encoded_sentences, encoded_labels, test_size=0.2, random_state=42)

In [28]:
class SentimentDataset(Dataset):
  def __init__(self, sentences, labels):
    self.sentences = torch.tensor(sentences, dtype=torch.long)
    self.labels = torch.tensor(labels, dtype=torch.long) # Changed to torch.long
  def __len__(self):
    return len(self.labels)
  def __getitem__(self, idx):
    return self.sentences[idx], self.labels[idx]


train_data = SentimentDataset(X_train[:int(len(X_train)*0.1)], y_train[:int(len(y_train)*0.1)])
test_data = SentimentDataset(X_test, y_test)

train_loader = DataLoader(train_data, batch_size=2, shuffle=True)
test_loader = DataLoader(test_data, batch_size=2)

In [24]:
 class SentimentClassifierRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(SentimentClassifierRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)  # Output_dim for multi-class classification

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.rnn(embedded)
        # For multi-class classification with CrossEntropyLoss, return raw logits
        output = self.fc(hidden.squeeze(0))
        return output

In [25]:
# Parameters
vocab_size = len(vocab) + 1  # +1 for padding
embedding_dim = 500
hidden_dim = 640
output_dim = 3  # Three classes: negative, neutral, positive

model = SentimentClassifierRNN(vocab_size, embedding_dim, hidden_dim, output_dim)
criterion = nn.CrossEntropyLoss()  # Use CrossEntropyLoss for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [26]:
train_data[4]

(tensor([    5,   572,     8,    14, 25223,    30,   838,     1,    79,    39,
             4,   864,    19,     1,   126,    38,    97,     6,   269,     9,
             5, 25224,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0]),
 tensor(1))

In [27]:
n_epochs = 1
for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    epoch_correct = 0
    total_samples = 0

    for sentences, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(sentences)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        # Calculate accuracy for multi-class classification
        predicted_labels = torch.argmax(outputs, dim=1)
        epoch_correct += (predicted_labels == labels).sum().item()
        total_samples += labels.size(0)

    # Calculate average loss and accuracy
    avg_loss = epoch_loss / len(train_loader)
    accuracy = epoch_correct / total_samples

    print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")

Epoch 1, Loss: 1.2462, Accuracy: 0.3471


In [29]:
# 6. Evaluation
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for sentences, labels in test_loader:
        outputs = model(sentences)
        # For multi-class, get the class with the highest probability (argmax)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate overall accuracy
accuracy = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {accuracy:.4f}")

# Optional: display a few predictions and their actual labels
# You can map the numeric predictions back to sentiment words if desired
reverse_label_mapping = {0: 'negative', 1: 'neutral', 2: 'positive'}
print("\nSample Predictions:")
for i in range(5):
    print(f"Actual: {reverse_label_mapping[all_labels[i]]}, Predicted: {reverse_label_mapping[all_preds[i]]}")

Test Accuracy: 0.4067

Sample Predictions:
Actual: positive, Predicted: neutral
Actual: negative, Predicted: neutral
Actual: neutral, Predicted: neutral
Actual: positive, Predicted: neutral
Actual: neutral, Predicted: neutral
